# 13 — Sequence Statistics

**Vignette index:** | [`01` GKMKernel basics](01_basic_kernel_matrix.ipynb) | [`02` Distance metrics & kernels](02_distance_metrics_and_kernels.ipynb) | [`03` SVM with kernel](03_svc_with_kernel.ipynb) | [`04` Clustering](04_clustering_sequences.ipynb) | [`05` Long sequence scoring](05_score_long_sequence.ipynb) | [`06` Weighted (WGKM) kernel](06_weighted_kernel.ipynb) | [`07` Transforms & comparison](07_transform_and_comparison.ipynb) | [`08` Windowed 3D tensors](08_windowed_3d_tensors.ipynb) | [`09` Spectrum encoder & NB](09_spectrum_encoder_and_differential.ipynb) | [`10` Gappy encoder](10_gappy_encoder.ipynb) | [`11` Mismatch encoder](11_mismatch_encoder.ipynb) | [`12` Shuffler & chunker](12_shuffler_and_chunker.ipynb) | `**13**` Sequence statistics

`kmer.stats` summarizes individual sequences: base composition, k-mer content, repetitiveness, and predictability. Every callable accepts a sequence or list of sequences and returns one row per sequence — directly usable as feature matrices for the models in kmer-learn.

In [1]:
from kmer.stats import (
    monomer_content, monomer_skew, kmer_content,
    windowed_monomer_content, homopolymer_stats,
    tandem_repeat_abundance, substring_complexity,
    lempel_ziv_complexity, markov_entropy, markov_transition_stats,
)

seqs = [
    "GCGCGCATATATGCGC",   # alternating repeats
    "AAAACCCCGGGGTTTT",   # blocky
    "ACGTTGCAATGCCGT",   # random-like
]

## Composition and skew

In [2]:
print("GC fraction:", monomer_content(seqs, letters="GC").ravel())
print("GC skew   :", monomer_skew(seqs, letters="GC").ravel())

GC fraction: [0.625      0.5        0.53333333]
GC skew   : [0. 0. 0.]


## K-mer content: entropy, Gini, duplicates

In [3]:
for stat in ("entropy", "gini", "duplicates"):
    print(f"{stat:10s}:", kmer_content(seqs, L=3, stat=stat).ravel())

entropy   : [0.74654742 0.84991455 0.9168498 ]
gini      : [0.91208791 0.95604396 0.97435897]
duplicates: [6. 4. 2.]


## Windowed summaries

Compute a statistic in windows along the sequence, then aggregate the profile (mean/std/median/extrema/quantiles/threshold counts).

In [4]:
region = "TTTTTTTTTT" + "GCGCGCGCGC" * 6 + "AAAAAACCCC"
summary = windowed_monomer_content(
    [region], window_size=10, step=5, agg=("mean", "std", "max"),
)
print("rows = sequences, columns = aggregations:")
print(summary)

rows = sequences, columns = aggregations:
[[0.82666667 0.3065217  1.        ]]


## Repetition and complexity

In [5]:
low = "AAAAACCCCCGGGGGTTTTTAAAAACCCCC"
high = "ACGTTGCAATGCCGTTACGATCGATCGTAC"

print("homopolymer (count, max):", homopolymer_stats([low, high], L_min=4, stat=("count", "max")))
print("tandem repeats            :", tandem_repeat_abundance(
    ["ACACACACGCGCGCGT", "ACGTTGCAACGTTGCA"],
    window_size=8, step=4, agg="count").ravel())
print("distinct substrings 1..8 :", substring_complexity([low, high], L_min=1, L_max=8).ravel())
print("Lempel-Ziv (normalized)  :", lempel_ziv_complexity([low, high]).ravel())

homopolymer (count, max): [[6. 5.]
 [0. 0.]]
tandem repeats            : [3. 0.]
distinct substrings 1..8 : [120. 153.]
Lempel-Ziv (normalized)  : [0.49068906 1.06315963]


## Markov diagnostics

Conditional entropy up to order 3, plus transition summaries. A deterministic sequence drops to zero entropy at order 1. These are per-sequence descriptors — not a generative background model; matched nulls come from `kmer.background`.

In [6]:
periodic = "ACACACACACACACAC"
random_like = "ACGTTGCAATGCCGTT"

print("order 0:", markov_entropy([periodic, random_like], order=0).ravel())
print("order 1:", markov_entropy([periodic, random_like], order=1).ravel())
print("order 2:", markov_entropy([periodic, random_like], order=2).ravel())

summary = markov_transition_stats([periodic, random_like], order=1)
print("transition summary:", summary.shape)

order 0: [0.5       0.9886085]
order 1: [0.         0.62516292]
order 2: [0.         0.07142857]
transition summary: (2, 7)
